<a href="https://colab.research.google.com/github/thansirmanoj-dotcom/mini-project/blob/main/fish_og_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q tensorflow opencv-python

import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import cv2
import json

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from google.colab import files

print(f'TensorFlow version: {tf.__version__}')
print('GPU available:', tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
print('Please upload your dataset zip file...')
uploaded = files.upload()

zip_filename = list(uploaded.keys())[0]
extract_path = '/content/fish_dataset'

with zipfile.ZipFile(zip_filename, 'r') as z:
    z.extractall(extract_path)

print(f'\nExtracted to: {extract_path}')
print('Contents:', os.listdir(extract_path))

Please upload your dataset zip file...


Saving archive (1).zip to archive (1).zip

Extracted to: /content/fish_dataset
Contents: ['train', 'valid', 'test']


In [ ]:
def find_split_folders(base):
    for split in ['train', 'valid', 'val', 'test']:
        if os.path.isdir(os.path.join(base, split)):
            return base
    for sub in os.listdir(base):
        sub_path = os.path.join(base, sub)
        if os.path.isdir(sub_path):
            for split in ['train', 'valid', 'val']:
                if os.path.isdir(os.path.join(sub_path, split)):
                    return sub_path
    return base

BASE_DIR = find_split_folders(extract_path)
print(f'Dataset root: {BASE_DIR}')
print('\nFolder structure:')
for folder in sorted(os.listdir(BASE_DIR)):
    folder_path = os.path.join(BASE_DIR, folder)
    if os.path.isdir(folder_path):
        subfolders = [f for f in os.listdir(folder_path) if os.path.isdir(os.path.join(folder_path, f))]
        files_count = sum([len(os.listdir(os.path.join(folder_path, s))) for s in subfolders]) if subfolders else len(os.listdir(folder_path))
        print(f'  {folder}/ → {files_count} items  |  Classes: {subfolders}')

TRAIN_DIR = os.path.join(BASE_DIR, 'train' if os.path.isdir(os.path.join(BASE_DIR, 'train')) else 'training')
VALID_DIR = os.path.join(BASE_DIR, 'valid' if os.path.isdir(os.path.join(BASE_DIR, 'valid')) else 'val')

CLASS_NAMES = sorted([d for d in os.listdir(TRAIN_DIR) if os.path.isdir(os.path.join(TRAIN_DIR, d))])
NUM_CLASSES = len(CLASS_NAMES)
print(f'\nClasses found ({NUM_CLASSES}): {CLASS_NAMES}')

Dataset root: /content/fish_dataset

Folder structure:
  test/ → 154 items  |  Classes: ['eye-fresh', 'gill-fresh', 'eye-non-fresh', 'gill-non-fresh']
  train/ → 3204 items  |  Classes: ['eye-fresh', 'gill-fresh', 'eye-non-fresh', 'gill-non-fresh']
  valid/ → 310 items  |  Classes: ['eye-fresh', 'gill-fresh', 'eye-non-fresh', 'gill-non-fresh']

Classes found (4): ['eye-fresh', 'eye-non-fresh', 'gill-fresh', 'gill-non-fresh']


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    shear_range=0.1,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

CLASS_LABELS = {v: k for k, v in train_generator.class_indices.items()}
print('\nClass index mapping:', train_generator.class_indices)
print('Label mapping:', CLASS_LABELS)

Found 3204 images belonging to 4 classes.
Found 310 images belonging to 4 classes.

Class index mapping: {'eye-fresh': 0, 'eye-non-fresh': 1, 'gill-fresh': 2, 'gill-non-fresh': 3}
Label mapping: {0: 'eye-fresh', 1: 'eye-non-fresh', 2: 'gill-fresh', 3: 'gill-non-fresh'}


In [ ]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,619,332 (9.99 MB)

 Trainable params: 361,348 (1.38 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,619,332 (9.99 MB)

 Trainable params: 361,348 (1.38 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
print('=== Phase 1: Feature Extraction ===')

base_model.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

print(f'\nBest feature extraction validation accuracy: {max(history.history["val_accuracy"]):.4f}')

print('\n=== Phase 2: Fine-tuning top layers ===')

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_ft = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

print(f'\nBest fine-tuned validation accuracy: {max(history_ft.history["val_accuracy"]):.4f}')

=== Phase 1: Feature Extraction ===


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 90s 694ms/step - accuracy: 0.5146 - loss: 0.9594 - val_accuracy: 0.7000 - val_loss: 0.5438 - learning_rate: 0.0010
Epoch 2/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 43s 422ms/step - accuracy: 0.7107 - loss: 0.5836 - val_accuracy: 0.7935 - val_loss: 0.4223 - learning_rate: 0.0010
Epoch 3/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 41s 408ms/step - accuracy: 0.7397 - loss: 0.5318 - val_accuracy: 0.7968 - val_loss: 0.3899 - learning_rate: 0.0010
Epoch 4/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 41s 408ms/step - accuracy: 0.7562 - loss: 0.5082 - val_accuracy: 0.7871 - val_loss: 0.4322 - learning_rate: 0.0010
Epoch 5/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 42s 409ms/step - accuracy: 0.7758 - loss: 0.4749 - val_accuracy: 0.8032 - val_loss: 0.4186 - learning_rate: 0.0010
Epoch 6/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 41s 407ms/step - accuracy: 0.7816 - loss: 0.4537 - val_accuracy: 0.8323 - val_loss: 0.3751 - learning_rate: 0.0010
Epoch 7/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 42s 417ms/step - accuracy: 0.8

In [ ]:
all_acc     = history.history['accuracy']     + history_ft.history['accuracy']
all_val_acc = history.history['val_accuracy'] + history_ft.history['val_accuracy']
all_loss    = history.history['loss']         + history_ft.history['loss']
all_val_loss= history.history['val_loss']     + history_ft.history['val_loss']

epochs_range  = range(len(all_acc))
phase2_start  = len(history.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, all_acc,     label='Train Accuracy', color='blue')
axes[0].plot(epochs_range, all_val_acc, label='Val Accuracy',   color='orange')
axes[0].axvline(x=phase2_start, color='gray', linestyle='--', label='Fine-tuning starts')
axes[0].set_title('Accuracy'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, all_loss,     label='Train Loss', color='blue')
axes[1].plot(epochs_range, all_val_loss, label='Val Loss',   color='orange')
axes[1].axvline(x=phase2_start, color='gray', linestyle='--', label='Fine-tuning starts')
axes[1].set_title('Loss'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
val_loss, val_accuracy = model.evaluate(val_generator, verbose=1)
print(f'\nValidation Loss:     {val_loss:.4f}')
print(f'Validation Accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)')

In [ ]:
def predict_fish_freshness(model, class_labels, img_size=(224, 224)):
    print('Upload a fish image for prediction...')
    uploaded_img = files.upload()

    if not uploaded_img:
        print('No file uploaded!')
        return

    img_path = list(uploaded_img.keys())[0]

    # Load and preprocess
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, img_size)
    img_normalized = img_resized / 255.0
    img_batch = np.expand_dims(img_normalized, axis=0)

    # Get softmax probabilities
    predictions = model.predict(img_batch, verbose=0)
    probs = predictions[0]

    # ── ARGMAX OUTPUT METHOD ──────────────────────────
    predicted_index = np.argmax(probs)
    predicted_label = class_labels[predicted_index]
    confidence      = probs[predicted_index]
    # ──────────────────────────────────────────────────

    # Show image + bar chart
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(f'Input: {os.path.basename(img_path)}')

    plt.subplot(1, 2, 2)
    colors = ['#2ecc71' if i == predicted_index else '#95a5a6' for i in range(len(class_labels))]
    bars = plt.barh([class_labels[i] for i in range(len(class_labels))], probs * 100, color=colors)
    plt.xlabel('Confidence (%)')
    plt.title('Softmax Probabilities (argmax → prediction)')
    plt.xlim(0, 100)
    for bar, prob in zip(bars, probs):
        plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{prob*100:.1f}%', va='center')
    plt.tight_layout()
    plt.show()

    # Print result
    print('=' * 45)
    print(f'  PREDICTION  : {predicted_label.upper()}')
    print(f'  CONFIDENCE  : {confidence*100:.2f}%')
    print(f'  METHOD      : argmax(softmax output)')
    print('=' * 45)
    print('\nAll class probabilities:')
    for i in range(len(class_labels)):
        marker = ' ← PREDICTED' if i == predicted_index else ''
        print(f'  {class_labels[i]:<20} : {probs[i]*100:6.2f}%{marker}')

    return predicted_label, float(confidence)

# Run it
result = predict_fish_freshness(model, CLASS_LABELS)

Upload a fish image for prediction...


In [ ]:
best_model.save('fish_freshness_model.keras')
print("Model saved as fish_freshness_model.keras")

# Uncomment to download:
# files.download('fish_freshness_model.keras')

In [ ]:
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

print('\n=== Phase 1: Feature Extraction ===')

base_model.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

print(f'\nBest feature extraction validation accuracy: {max(history.history["val_accuracy"]):.4f}')

print('\n=== Phase 2: Fine-tuning top layers ===')

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_ft = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

print(f'\nBest fine-tuned validation accuracy: {max(history_ft.history["val_accuracy"]):.4f}')